In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import pandas as pd
from pyspark.sql import SparkSession

configs = [
    {
        "file": "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/EIOPA_RFR_20251231_Term_Structures.xlsx",
        "sheet": "RFR_spot_no_VA",
        "version_id": "RFR_20251231_noVA"
    },
    {
        "file": "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/EIOPA_RFR_20251231_Term_Structures.xlsx",
        "sheet": "RFR_spot_with_VA",
        "version_id": "RFR_20251231_withVA"
    },
    {
        "file": "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/EIOPA_RFR_20260331_Term_Structures.xlsx",
        "sheet": "RFR_spot_no_VA",
        "version_id": "RFR_20260331_noVA"
    },
    {
        "file": "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/EIOPA_RFR_20260331_Term_Structures.xlsx",
        "sheet": "RFR_spot_with_VA",
        "version_id": "RFR_20260331_withVA"
    },
]

all_curves = []

for cfg in configs:
    df = pd.read_excel(cfg["file"], sheet_name=cfg["sheet"], header=1)
    
    tenor_col = df.columns[1]
    euro_col = df.columns[2]
    
    df_filtered = df[df[tenor_col].apply(lambda x: str(x).isdigit())]
    
    df_curve = df_filtered[[tenor_col, euro_col]].copy()
    df_curve.columns = ["tenor_year", "zero_rate_annual"]
    
    df_curve["tenor_year"] = df_curve["tenor_year"].astype(int)
    df_curve["zero_rate_annual"] = df_curve["zero_rate_annual"].astype(float)
    df_curve["tenor_month"] = df_curve["tenor_year"] * 12
    df_curve["version_id"] = cfg["version_id"]
    
    all_curves.append(df_curve)
    print(f"✅ {cfg['version_id']}: {len(df_curve)} rows")

df_all = pd.concat(all_curves, ignore_index=True)
print(f"\n📊 Total: {len(df_all)} rows, {df_all['version_id'].nunique()} versions")
print(df_all.groupby("version_id").size())



In [0]:
spark = SparkSession.builder.getOrCreate()
df_spark = spark.createDataFrame(df_all)

(
    df_spark
    .select("tenor_month", "zero_rate_annual", "version_id")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.life_insurance_raw.discount_curve")
)

display(df_spark.select("version_id", "tenor_month", "zero_rate_annual"))